In [7]:
import spacy
import pandas as pd
from pathlib import Path
import csv

In [8]:
def extract_perfect_constructions(
    file_path: str,
    lang_code: str,
    output_csv: str = None,
    tokenize_gloss: bool = True
) -> pd.DataFrame:
    """
    Extracts perfect-like constructions (AUX 'have' + VERB) from a glossed bilingual Bible file.

    Parameters:
    - file_path: str, path to the glossed corpus file
    - lang_code: str, language code used in gloss lines (e.g., "AYR")
    - output_csv: str, optional path to save output CSV
    - tokenize_gloss: bool, if True, split glossed line into separate columns

    Returns:
    - pd.DataFrame with filtered aligned data
    """
    # Load spaCy English model
    nlp = spacy.load("en_core_web_sm")

    # Read glossed file
    with open(file_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f.readlines()]

    # Identify ENG lines and their indices
    eng_lines = [(idx, line.replace("ENG:", "").strip()) for idx, line in enumerate(lines) if line.startswith("ENG:")]

    # Detect perfect-like constructions
    matching_indices = []
    for idx, sentence in eng_lines:
        doc = nlp(sentence)
        for i, token in enumerate(doc):
            if token.pos_ == "AUX" and token.lemma_ == "have":
                for j in range(i + 1, len(doc)):
                    if doc[j].pos_ in {"VERB", "AUX"} and doc[j].tag_ == "VBN":
                        matching_indices.append(idx)
                        break
                break  # only look for first HAVE per sentence

    # Extract verse blocks using the original lines
    verse_blocks = []
    for eng_idx in matching_indices:
        verse_idx = eng_idx - 1
        gloss_idx = eng_idx + 1
        if verse_idx < 0 or gloss_idx >= len(lines):
            continue

        verse_line = lines[verse_idx].strip()
        eng_line = lines[eng_idx].strip()
        gloss_line = lines[gloss_idx].strip()

        if not (verse_line.startswith("verse:") and eng_line.startswith("ENG:") and gloss_line.startswith(f"{lang_code}:")):
            continue

        verse_id = verse_line.replace("verse:", "").strip()
        eng_clean = eng_line.replace("ENG:", "").strip()
        gloss_clean = gloss_line.replace(f"{lang_code}:", "").strip()

        if tokenize_gloss:
            gloss_tokens = gloss_clean.split()
            row = [verse_id, eng_clean] + gloss_tokens
        else:
            row = [verse_id, eng_clean, gloss_clean]

        verse_blocks.append(row)

    # Normalize row length
    max_len = max(len(row) for row in verse_blocks)
    for row in verse_blocks:
        row += [""] * (max_len - len(row))

    # Column headers
    if tokenize_gloss:
        columns = ["verse_id", "english"] + [f"gloss_{i}" for i in range(max_len - 2)]
    else:
        columns = ["verse_id", "english", "gloss"]

    df = pd.DataFrame(verse_blocks, columns=columns)

    # Optional CSV export
    if output_csv:
        df.to_csv(output_csv, index=False, encoding="utf-8")
        print(f"✅ Output written to {output_csv}")

    return df

# Example usage:
# df = extract_perfect_constructions("ayr-x-bible-1997_glossed.txt", "AYR", output_csv="ayr_perfects.csv")

In [9]:
extract_perfect_constructions("/Users/glossophilia/Desktop/Data/Bibs/Glossed/deu_glossed_corrected.txt", "DEU", output_csv="deu_perfects.csv")

✅ Output written to deu_perfects.csv


,verse_id,english,gloss_0,gloss_1,gloss_2,gloss_3,gloss_4,gloss_5,gloss_6,gloss_7,...,gloss_81,gloss_82,gloss_83,gloss_84,gloss_85,gloss_86,gloss_87,gloss_88,gloss_89,gloss_90
0,1001031,God saw all that he had made . It was very goo...,und/,gott/GOD,sah/SEE,alles/ALL,an/,",/",was/THAT,er/,...,,,,,,,,,,
1,1002002,By the seventh day God had finished his work ....,und/,gott/GOD,vollendete/FINISH,am/BY_THE,siebenten/SEVENTH,tage/DAY,sein/,werk/WORK,...,,,,,,,,,,
2,1002003,God blessed the seventh day and made it holy ....,und/,gott/GOD,segnete/BLESS,den/THE,siebenten/SEVENTH,tag/DAY,und/AND,heiligte/MAKE_HOLY_THIS,...,,,,,,,,,,
3,1002005,No shrub of the field had yet appeared on the ...,noch/,gab/,es/,aber/,kein/NO,gesträuch/SHRUB,des/,feldes/FIELD,...,,,,,,,,,,
4,1002008,"Now Jehovah God planted a garden in the east ,...",dann/,pflanzte/PLANT,gott/GOD,der/,herr/JEHOVAH,einen/A,garten/GARDEN,in/IN,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4446,66019020,"The beast was captured , and with him the fals...",und/,das/THE,tier/BEAST,wurde/BE,ergriffen/CAPTURE,und/AND,mit/WITH,ihm/-PRON-,...,,,,,,,,,,
4447,66020004,"I saw thrones , and they sat on them , and jud...",und/,ich/-PRON-,sah/SEE,throne/THRONE,",/,",und/AND,sie/-PRON-,setzten/SIT,...,,,,,,,,,,
4448,66021001,I saw a new heaven and a new earth : for the f...,und/,ich/,sah/SEE,einen/A,neuen/NEW,himmel/HEAVEN,und/AND,eine/A,...,,,,,,,,,,
4449,66021004,""" God will wipe away all tears from their eyes...",und/,er/,wird/WILL,alle/ALL,tränen/TEAR,abwischen/WIPE,von/FROM,ihren/-PRON-,...,,,,,,,,,,
